# MCP + LangChain Lab

## Step 1: Setup and installation

This notebook is the executable entry point for the lab. It installs the required packages, loads the local `.env` file, and verifies that the OpenAI API key is available without printing the secret.

In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


The next cell checks the environment. Run the notebook with the `bootcamp-env` kernel, or another Python 3.10+ environment.

In [2]:
import importlib.metadata
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

REQUIRED_DISTRIBUTIONS = (
    "langchain",
    "langchain-openai",
    "langchain-mcp-adapters",
    "mcp",
    "python-dotenv",
)

if sys.version_info < (3, 10):
    raise RuntimeError("Use Python 3.10 or newer for this lab.")

load_dotenv(Path.cwd() / ".env")
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY was not found. Add it to .env or export it in the shell."
    )

print(f"Python: {sys.version.split()[0]}")
print("OPENAI_API_KEY: loaded (value hidden)")
print("Required packages:")
for distribution in REQUIRED_DISTRIBUTIONS:
    print(f"  - {distribution}: {importlib.metadata.version(distribution)}")

Python: 3.11.16
OPENAI_API_KEY: loaded (value hidden)
Required packages:
  - langchain: 1.4.7
  - langchain-openai: 1.7.3
  - langchain-mcp-adapters: 0.3.2
  - mcp: 1.30.0
  - python-dotenv: 1.2.4


Step 1 is complete when the install cell succeeds, the package versions are printed, and the API key is reported as loaded.

## Step 2: Connect to an MCP server

This step connects to the official LangChain documentation MCP server. It is a remote server, so the client uses HTTP and does not start a local process. The server is read-only for this lab and does not require another secret.

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "langchain-docs": {
            "transport": "http",
            "url": "https://docs.langchain.com/mcp",
        }
    }
)

print("MCP client configured")
print("Server: langchain-docs")
print("Transport: HTTP")
print("URL: https://docs.langchain.com/mcp")

MCP client configured
Server: langchain-docs
Transport: HTTP
URL: https://docs.langchain.com/mcp


Creating the client stores the connection details. The next cell makes the first request to the server and verifies that the connection works by discovering its tools and resources.

In [4]:
mcp_tools = await mcp_client.get_tools()

print(f"Discovered {len(mcp_tools)} tool(s):")
for tool in mcp_tools:
    print(f"  - {tool.name}")

mcp_resources = await mcp_client.get_resources()
print(f"\nDiscovered {len(mcp_resources)} resource(s):")
for index, resource in enumerate(mcp_resources, start=1):
    print(f"  - {index}: {getattr(resource, 'source', type(resource).__name__)}")

Discovered 3 tool(s):
  - search_docs_by_lang_chain
  - query_docs_filesystem_docs_by_lang_chain
  - submit_feedback

Discovered 1 resource(s):
  - 1: None


Step 2 is complete when the discovery cell returns the server's tool list and resource list. The client and discovered objects will be reused in the next steps.

## Step 3: Load MCP tools into LangChain

`MultiServerMCPClient.get_tools()` has already converted the MCP definitions into LangChain-compatible tools. This step gives that list an explicit name and inspects each tool so we can verify what an agent will receive.

In [5]:
# The result of get_tools() is already a list of LangChain BaseTool objects.
langchain_tools = mcp_tools

print(f"Loaded {len(langchain_tools)} LangChain tool(s) from the MCP server:")
for tool in langchain_tools:
    print(f"\n- {tool.name}")
    print(f"  type: {type(tool).__name__}")
    print(f"  description: {tool.description}")
    print(f"  input schema: {tool.args}")

Loaded 3 LangChain tool(s) from the MCP server:

- search_docs_by_lang_chain
  type: StructuredTool
  description: Search across the Docs by LangChain knowledge base to find relevant information, code examples, API references, and guides. Use this tool when you need to answer questions about Docs by LangChain, find specific documentation, understand how features work, or locate implementation details. The search returns contextual content with titles and direct links to the documentation pages. If you need the full content of a specific page, use the query_docs_filesystem tool to `head` or `cat` the page path (append `.mdx` to the path returned from search — e.g. a result at `/some/page` is read with `head -200 /some/page.mdx`).
  input schema: {'query': {'type': 'string', 'description': 'Search query'}}

- query_docs_filesystem_docs_by_lang_chain
  type: StructuredTool
  description: Run a read-only shell-like query against a virtualized, in-memory filesystem rooted at `/` that contai

The tool names, descriptions, and input schemas are now available to pass directly to `create_agent` in Step 4.

## Step 4: Create an agent with MCP tools

The agent receives the LangChain tools from Step 3. Its instructions require it to search the official documentation before answering, so the response can be traced to an MCP tool result instead of relying on model memory.

In [6]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = create_agent(
    model=llm,
    tools=langchain_tools,
    system_prompt=(
        "You answer questions about LangChain using the official documentation MCP tools. "
        "Always call an MCP documentation tool before answering. Do not guess or rely on memory. "
        "Explain the answer clearly and distinguish documented facts from uncertainty."
    ),
)

print(f"Agent created with {len(langchain_tools)} MCP tool(s).")

Agent created with 3 MCP tool(s).


In [7]:
question = "According to the official LangChain documentation, how do I create an agent with tools in Python?"
print(f"Question: {question}\n")

result = await agent.ainvoke({"messages": [HumanMessage(content=question)]})

print("Tool evidence:")
for message in result["messages"]:
    for call in getattr(message, "tool_calls", []):
        print(f"  called: {call['name']} with {call['args']}")
    if getattr(message, "type", "") == "tool":
        print(f"  returned: {str(message.content)[:500]}...")

print("\nFinal answer:")
print(result["messages"][-1].content)

Question: According to the official LangChain documentation, how do I create an agent with tools in Python?

Tool evidence:
  called: search_docs_by_lang_chain with {'query': 'create an agent with tools in Python'}
  returned: [{'type': 'text', 'text': 'Title: Tools\nLink: https://docs.langchain.com/oss/python/deepagents/tools\nPage: oss/python/deepagents/tools\nContent: Deep Agents can call any tool you define, any [LangChain tool](/oss/python/langchain/tools), and tools from any [MCP server](#mcp-tools).\nPass them to `create_deep_agent` via the `tools=` parameter alongside the [built-in harness tools](/oss/python/deepagents/overview#execution-environment) for file management and subagent spawning.\n\n<CodeGroup>\n ...

Final answer:
To create an agent with tools in Python using LangChain, you can follow these steps based on the official documentation:

1. **Define Your Tools**: You can define tools using the `@tool` decorator from `langchain.tools`. Each tool should be a callable fu

Step 4 is complete when the agent makes at least one MCP tool call and produces a final answer grounded in the returned documentation.

## Step 5: Access MCP resources

An MCP resource is read-only context that the application fetches and gives to the model. Unlike a tool, the agent does not call the resource itself. We will read the resource returned by `get_resources()` and place its text in a separate agent's system prompt.

In [8]:
resource_text = mcp_resources[0].as_string()

print(f"Read {len(resource_text):,} characters from the first MCP resource.")
print("Resource preview:")
print(resource_text[:500])

Read 10,130 characters from the first MCP resource.
Resource preview:
---
name: langchain
description: "Use when building AI agents with tool calling, designing
  multi-step workflows, integrating LLMs with external systems, or deploying
  production agents. Agents are the core pattern: a model calling tools in a
  loop until a task completes, shaped by a configurable harness of prompt,
  tools, and middleware."
metadata:
  mintlify-proj: langchain
  version: "1.0"
---

# LangChain Skill

## Product Summary

LangChain is a framework for building agents—systems whe


In [9]:
guide_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "Answer only from the following LangChain MCP resource. "
        "If the resource does not contain the answer, say so clearly.\n\n"
        f"RESOURCE CONTENT:\n{resource_text}"
    ),
)

question = "What do I need to add to an agent so that it remembers the conversation between calls?"
guide_result = await guide_agent.ainvoke({"messages": [HumanMessage(content=question)]})
answer = guide_result["messages"][-1].content

print(f"Question: {question}\n")
print("Resource-grounded answer:")
print(answer)

evidence_keyword = "thread_id"
print(
    f"\nEvidence check: {evidence_keyword!r} appears in the resource: "
    f"{evidence_keyword in resource_text}"
)

Question: What do I need to add to an agent so that it remembers the conversation between calls?

Resource-grounded answer:
To enable an agent to remember the conversation between calls, you need to use middleware for memory. Specifically, you can add `MemoryMiddleware` to your agent. This middleware allows the agent to persist state across messages, enabling it to recall previous interactions during the conversation. 

Additionally, you should ensure that you pass a `thread_id` in the configuration to scope the conversation and maintain context.

Evidence check: 'thread_id' appears in the resource: True


Step 5 is complete when the agent answers from the injected resource text and the evidence check confirms that a key term from the answer is present in that resource. No MCP tool-call log is expected here because the application, not the agent, fetched the resource.

## Step 6: Build a complete MCP-enabled agent

This agent combines the MCP tools and MCP resource context in one workflow. Multiple MCP servers are optional, so this implementation keeps one server and demonstrates both tools and resources together.

In [10]:
complete_agent = create_agent(
    model=llm,
    tools=langchain_tools,
    system_prompt=(
        "You are a LangChain expert. Use the MCP documentation tools to verify claims before answering. "
        "Use the resource background as additional context, but do not invent information. "
        "When you use a tool, explain which documentation result supports the answer.\n\n"
        f"RESOURCE BACKGROUND:\n{resource_text}"
    ),
)

print("Complete agent created with MCP tools and resource context.")

Complete agent created with MCP tools and resource context.


In [11]:
question = (
    "Using the official LangChain documentation, explain how to create an agent with tools, "
    "and identify the configuration value used to keep separate conversations apart."
)
print(f"Question: {question}\n")

complete_result = await complete_agent.ainvoke(
    {"messages": [HumanMessage(content=question)]}
)

print("Tool evidence:")
for message in complete_result["messages"]:
    for call in getattr(message, "tool_calls", []):
        print(f"  called: {call['name']} with {call['args']}")
    if getattr(message, "type", "") == "tool":
        print(f"  returned: {str(message.content)[:500]}...")

print("\nFinal answer:")
print(complete_result["messages"][-1].content)

Question: Using the official LangChain documentation, explain how to create an agent with tools, and identify the configuration value used to keep separate conversations apart.

Tool evidence:
  called: search_docs_by_lang_chain with {'query': 'create an agent with tools'}
  called: search_docs_by_lang_chain with {'query': 'configuration value to keep separate conversations apart'}
  returned: [{'type': 'text', 'text': 'Title: Use MCP tools in an agent\nLink: https://docs.langchain.com/oss/python/langchain/mcp/tools#use-mcp-tools-in-an-agent\nPage: oss/python/langchain/mcp/tools\nContent: ## Use MCP tools in an agent\n\nDiscover the server\'s catalog with [`MCPAdapter.list_tools`](https://reference.langchain.com/python/langchain/mcp/adapter/MCPAdapter/list_tools), then give the returned tools to [`create_agent`](https://reference.langchain.com/python/langchain/agents/factory/create_ag...
  returned: [{'type': 'text', 'text': 'Title: Combine with thread isolation\nLink: https://docs.lan

Step 6 is complete when one agent can use the MCP tools, receive resource context, and produce a traceable answer.